In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

%matplotlib inline

In [2]:
df = pd.read_csv("/home/ashish/Desktop/All/Algorithms/10_Gradiant_Boost/cardekho_imputated.csv", index_col=[0])

In [3]:
## Check Null Values
##Check features with nan value
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [4]:
# removing unnececry data 
df.drop('car_name',axis=1,inplace=True)
df.drop('brand',axis=1,inplace=True)

In [5]:
df['model'].unique()

<StringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [6]:
# Getting all diffrennt parnameters of fetures
num_features = [feature for  feature in df.columns if df[feature].dtype != 'int']
print('Num Of Numerical Features :',len(num_features))

cat_features = [data for data in df.columns if df[data].dtype == 'str']
print("Num of Categrical Feature :", len(cat_features))

discrete_features = [data for data in num_features if len(df[data].unique())<=25]
print('Num of Discrete feature  :',len(discrete_features))

Num Of Numerical Features : 6
Num of Categrical Feature : 4
Num of Discrete feature  : 3


In [7]:
## Getting All Different Types OF Features
num_features = [feature for feature in df.columns if df[feature].dtype != 'int']
print('Num of Numerical Features :', len(num_features))

cat_features = [feature for feature in df.columns if df[feature].dtype == 'str']
print('Num of Categorical Features :', len(cat_features))

discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features :',len(discrete_features))

continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

Num of Numerical Features : 6
Num of Categorical Features : 4
Num of Discrete Features : 3
Num of Continuous Features : 3


In [8]:
## Indpendent and dependent features
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [9]:
len(df['model'].unique())

120

In [10]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [11]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [12]:
# for labeling in the Model columns for categrical data 
X['model']=le.fit_transform(X['model'])

In [13]:
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

In [14]:
## Devidind the data according to the categries 
cat_features = ['seller_type','fuel_type','transmission_type'] 
num_features =X.select_dtypes(exclude='object').columns  # numeric data

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

scaler_transform = StandardScaler()
oh_transform = OneHotEncoder(drop='first')

procceses = ColumnTransformer([

    ("Oh",oh_transform,cat_features),
    ("Scaler",scaler_transform,num_features)

],remainder='passthrough'
)

In [15]:
X = procceses.fit_transform(X)

In [16]:
## Now i will spliting the train and test
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((12328, 14), (3083, 14))

In [17]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression,Ridge, Lasso
from  sklearn.ensemble import GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [18]:
##Create a Function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mean_squared_error(true, predicted))
    r2_square = r2_score(true, predicted)
    return mae, rmse, r2_square

In [19]:
## Bigigng the Model training
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "Adaboost":AdaBoostRegressor()
   
}

for i in range (len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train,y_train)

    ## Make Prediction
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    ## Evalution of dataset
    model_train_mae, model_train_rmse, model_train_r2 = evaluate_model(y_train,y_pred_train)

    model_test_mae, model_test_rmse, model_test_r2 = evaluate_model(y_test,y_pred_test)

    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Root Mean Squared Error: 553855.6665
- Mean Absolute Error: 268101.6071
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502543.5930
- Mean Absolute Error: 279618.5794
- R2 Score: 0.6645


Lasso
Model performance for Training set
- Root Mean Squared Error: 553855.6710
- Mean Absolute Error: 268099.2226
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502542.6696
- Mean Absolute Error: 279614.7461
- R2 Score: 0.6645


Ridge
Model performance for Training set
- Root Mean Squared Error: 553856.3160
- Mean Absolute Error: 268059.8015
- R2 Score: 0.6218
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 502533.8230
- Mean Absolute Error: 279557.2169
- R2 Score: 0.6645


Gradient Boosting
Model performance for Training set
- Root Mean Squared Error: 204944.5104
- Mean Abso

In [ ]:
## Hyperparameter Tuning 
rf_parms = {
    'max_depth':[3,4,5,8,10,12,None],
    'max_features':[3,5,8,7],
    'min_samples_split':[2,8,15,20,4],
    'n_estimators':[100,200,500,1000]
}

gradient_params = {
    'n_estimators'    : [100, 200, 500],
    'learning_rate'   : [0.01, 0.05, 0.1],
    'max_depth'       : [3, 5, 7,8,12],
    'colsample_bytree': [0.7, 1.0,0.3,0.4]
}


In [21]:
## Model list for Hyper perameter Tunning
randomcv_model = [

                  ("RF", RandomForestRegressor(), rf_parms),
                  ("GD", GradientBoostingRegressor(), gradient_params)

]

In [22]:
X_train.shape , y_train.shape

((12328, 14), (12328,))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

model_parms = {}
for name,model,parms in randomcv_model:
    random = RandomizedSearchCV(estimator=model,
                                param_distributions=parms,
                                n_iter=100,
                                cv=3,
                                verbose=2,
                                n_jobs=1)
    
    random.fit(X_train,y_train)
    model_parms[name] = random.best_params_

for model_name in model_parms:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_parms[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END max_depth=8, max_features=8, min_samples_split=20, n_estimators=1000; total time=   5.5s
[CV] END max_depth=8, max_features=8, min_samples_split=20, n_estimators=1000; total time=   5.4s
[CV] END max_depth=8, max_features=8, min_samples_split=20, n_estimators=1000; total time=   5.5s
[CV] END max_depth=8, max_features=5, min_samples_split=2, n_estimators=200; total time=   0.8s
[CV] END max_depth=8, max_features=5, min_samples_split=2, n_estimators=200; total time=   0.8s
[CV] END max_depth=8, max_features=5, min_samples_split=2, n_estimators=200; total time=   0.8s
[CV] END max_depth=8, max_features=5, min_samples_split=8, n_estimators=100; total time=   0.4s
[CV] END max_depth=8, max_features=5, min_samples_split=8, n_estimators=100; total time=   0.4s
[CV] END max_depth=8, max_features=5, min_samples_split=8, n_estimators=100; total time=   0.4s
[CV] END max_depth=3, max_features=7, min_samples_split=20, n_estim

In [26]:
## Retrain the model with parms
models = {
    "Random Forest Reegression": RandomForestRegressor(n_estimators=100,# Zyada trees
                                                       min_samples_split=2,
                                                       max_features=7,
                                                       max_depth=None,
                                                       n_jobs=-1),
    "Gradient-Boosting" :  GradientBoostingRegressor(n_estimators=200,
                                                     min_samples_split= 2, max_depth= 5, learning_rate= 0.1 )                                                   
                                                                                                                                                               
}

for i in range(len(list(models))):
    model=list(models.values())[i]
    model.fit(X_train,y_train)

     # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae , model_test_rmse, model_test_r2 = evaluate_model(y_test, y_test_pred)
    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    print("- Root Mean Squared Error: {:.4f}".format(model_train_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_train_mae))
    print("- R2 Score: {:.4f}".format(model_train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Root Mean Squared Error: {:.4f}".format(model_test_rmse))
    print("- Mean Absolute Error: {:.4f}".format(model_test_mae))
    print("- R2 Score: {:.4f}".format(model_test_r2))
    
    print('='*35)
    print('\n')


Random Forest Reegression
Model performance for Training set
- Root Mean Squared Error: 118842.9637
- Mean Absolute Error: 39166.9738
- R2 Score: 0.9826
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 213586.2204
- Mean Absolute Error: 98764.3027
- R2 Score: 0.9394


Gradient-Boosting
Model performance for Training set
- Root Mean Squared Error: 110967.8599
- Mean Absolute Error: 74260.8858
- R2 Score: 0.9848
----------------------------------
Model performance for Test set
- Root Mean Squared Error: 217127.3290
- Mean Absolute Error: 98604.4791
- R2 Score: 0.9374


